> **Research-source reading copy.** Saved outputs and machine-specific metadata were removed for sharing. These original workflow cells require the private raw/source-evidence workspace and database, which are intentionally not bundled. For an executable portable result replay, use `python scripts/replay_saved_study.py` from the repository root. See README and docs/REPRODUCIBILITY.md.

# USD/PHP static stress testing — safe reruns

**Question:** How would specified exchange-rate movements change a fixed position's peso value?

Six hypothetical shocks and one historical percentage decline complement the frozen VaR study. The hypothetical long USD1 million position and BPI's dated supporting exposure remain distinct. No probability, forecast horizon or bank risk limit is assigned.

The verified `reports/stress_testing_v1` bundle remains unchanged. New executions publish unique validated folders. Read `STRESS_TESTING_CONTRACT.md` and `STRESS_RERUN_SAFEGUARDS.md` for assumptions and execution rules.

In [ ]:
from pathlib import Path
import sys, importlib, io
import pandas as pd
from IPython.display import display, Image

project_root = Path.cwd().resolve()
if not (project_root / 'STRESS_TESTING_CONTRACT.md').is_file():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'scripts'))
import stress_safeguards as stress
stress = importlib.reload(stress)

# Fixed inputs: an alteration requires a new methodology decision.
config = stress.StressConfig(usd_position=1_000_000,
                            hypothetical_percent=(-5, -3, -1, 1, 3, 5))
session = stress.prepare_inputs(project_root, config)
print('Approved input hashes and all frozen validation files verified.')
print('Existing stress bundle remains read-only.')

## 1. Establish the valuation and historical replay

The latest frozen numeric endpoint is **62.485 PHP/USD on 30 September 2026**. Select the minimum exact return over all1,647 original intervals, including review intervals. Ties use earliest end date then original interval order; no filling or bridging.

The exact workbook identity and three date/rate cells are checked. Existing source/audit evidence is reused with its status and hashes. Source dates are not verified market-close or publication timestamps. The historical percentage is replayed on the September2026 baseline.

In [ ]:
selection = stress.select_market(session)
historical = selection['historical_interval']
display(pd.DataFrame([{'starting_date': selection['starting_source_date'],
                      'PHP_per_USD': selection['starting_rate_raw'],
                      'fixed_USD_position': config.usd_position}]))
display(pd.DataFrame([{k: historical[k] for k in (
    'start_source_date', 'end_source_date', 'start_rate_raw', 'end_rate_raw',
    'gap_calendar_days', 'calendar_eligibility', 'interval_id')}]))

## 2. Assess the separate BPI case

BPI's input is the **positive PHP814 million consolidated Group net USD on-balance-sheet exposure as of 31 December 2025**, already in peso equivalent. It is not USD814 million, the Parent Bank, a daily series or total economic risk.

Working provenance, PDF identity, scope, date, units, signed net and comparator fields must match approved evidence. **If eligibility fails, withhold BPI values and still complete USD-only results.** Original publication date remains unknown.

In [ ]:
bpi_status = stress.assess_bpi(session)
print('BPI:', 'eligible for static illustration' if bpi_status['eligible'] else 'WITHHELD')
if bpi_status['eligible']:
    print('Comparator: PHP14,489,200 versus approximately PHP14 million.')
    print('Difference PHP489,200 fits the fixed PHP500,000 rounding allowance.')
else:
    print('Reason:', bpi_status['reason'])
    print('The USD1 million case continues without BPI scenario values.')

## 3. Apply scenarios and check known answers

USD P&L is the ending value minus the starting value of an unchanged holding. Signed loss is its negative. Exact fractions, direct revaluation and decimal arithmetic must agree. Literal known answers catch wrong prescribed quantities/shocks that shared-input identities could miss.

Zero-shock controls follow the same record/export pipeline but remain outside the seven scenarios. Ending USD valuations are explicitly exported. BPI, when eligible, uses its separately dated peso-equivalent exposure without a September FX conversion.

In [ ]:
model = stress.calculate(session, selection, bpi_status)
results = pd.DataFrame(model['records'])
controls = pd.DataFrame(model['controls'])
comparison = results.pivot(index='scenario', columns='case', values='pnl_php')
comparison = comparison.reindex(list(dict.fromkeys(results['scenario'])))
display(comparison.style.format('PHP{:,.2f}'))
display(controls[['case', 'scenario', 'pnl_php', 'ending_position_value_php']])
print(f"Passed inputs, scenario membership, signs and literal controls: {len(results)} scenario rows.")

## 4. Read the impact chart

Left of zero means hypothetical loss; right means gain. Each case has its own scale and date. Labels come from validated scenario records. When BPI is withheld, only the USD panel appears. Exposure size does not establish a bank risk ranking.

In [ ]:
figure = stress.make_chart(model)
chart_buffer = io.BytesIO()
figure.savefig(chart_buffer, format='png', dpi=140, bbox_inches='tight')
display(Image(data=chart_buffer.getvalue()))
import matplotlib.pyplot as plt
plt.close(figure)
print('Chart displayed; original outputs were not overwritten.')

## 5. Findings, limits and safe publication

- USD1 million starts at **PHP62.485 million**. The -1/-3/-5% scenario losses are PHP624,850 /1,874,550 /3,124,250 under fixed linear assumptions. Opposite shocks give opposite P&L.
- The original4–5 August2025 interval had a **hypothetical USD1 million valuation loss of PHP878,000**. Replaying its exact percentage on the September2026 baseline differs because the starting value differs. Neither is evidence of an actual bank position or realised loss.
- BPI's static mapping, **only if eligible**, changes by PHP8.14 million per1%. Withheld BPI values are not presented as verified results.
- No probability, forecast horizon, avoided-loss benefit, maximum-loss bound or actual BPI loss is established. Stress testing does not repair VaR coverage. Hedge sensitivities, nonlinear instruments, liquidity, costs and behavioural responses remain outside scope.

Exports are staged, read back, and checked against protected inputs before a unique run directory is published. Read outputs only when `SUCCESS.json` and its checksum manifest verify. No prior run is overwritten. Implementation is adviser-assisted; source timing and original BPI publication date remain qualified.

In [ ]:
run_path = stress.publish_run(session, model)
run_status = stress.verify_run(run_path)
print('Validated run:', run_path)
print('Status:', run_status['status'])
print('Exported exact P&Ls, ending USD valuations, zero controls, settings and hashes.')
print('Original stress v1, frozen VaR files, approved core inputs and database remain unchanged.')